# CONDA toxicity detection — PATCHED build — context window k=0

This notebook fine-tunes **DeBERTa-v3-base** on the CONDA dataset with **k=0 previous messages** as conversational context.

**Output:** a saved model directory on Google Drive that can be loaded by the demo website.

**Runtime:** ~75 min on a Colab T4 GPU. Set `Runtime > Change runtime type > GPU` before running.

## 1. Setup — install deps, mount Drive

In [ ]:
# Install dependencies (Colab usually has torch + transformers; sentencepiece is needed for DeBERTa-v3)
!pip install -q sentencepiece protobuf

In [ ]:
from pathlib import Path

K = 0                                          # ← change : 0, 1, 3, 5, ou 10 selon le notebook

# Local Colab disk (NOT Drive). Will be lost when the session ends — make sure
# to download the .tar.gz at the end of the notebook.
SAVE_ROOT = Path('/content/conda_ksweep')
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = SAVE_ROOT / f'k{K}_model'
CKPT_DIR  = SAVE_ROOT / f'k{K}_checkpoints'
MODEL_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)
print(f'Will save model to: {MODEL_DIR}')
print(f'Training checkpoints in : {CKPT_DIR}')

## 1.5 · LayerNorm legacy-name patch  (CRITICAL — fixes DeBERTa-v3 bug)

The original `microsoft/deberta-v3-base` checkpoint stores LayerNorm parameters
under the legacy names `beta` / `gamma`. Modern `transformers` (>= 4.40) renamed
them to `weight` / `bias`. The automatic re-mapping fails in transformers v5.x
for DeBERTa-v3 specifically, leaving the LayerNorm params randomly initialised.

This makes the model train OK in memory (the classifier head compensates) but
the saved checkpoint collapses to predicting the majority class because the
re-mapping fails again on every reload.

The patch below intercepts state-dict loading and remaps the legacy keys so
LayerNorm parameters survive every save/load round-trip.


In [ ]:
# === DeBERTa-v3 LayerNorm legacy-key patch ===
import torch.nn as nn

_old_load = nn.LayerNorm._load_from_state_dict

def _patched_load_from_state_dict(self, state_dict, prefix, local_metadata,
                                  strict, missing_keys, unexpected_keys, error_msgs):
    # Map legacy beta/gamma -> bias/weight before letting the original loader run.
    for old, new in [("beta", "bias"), ("gamma", "weight")]:
        old_key = prefix + old
        new_key = prefix + new
        if old_key in state_dict and new_key not in state_dict:
            state_dict[new_key] = state_dict.pop(old_key)
    return _old_load(self, state_dict, prefix, local_metadata,
                     strict, missing_keys, unexpected_keys, error_msgs)

nn.LayerNorm._load_from_state_dict = _patched_load_from_state_dict
print('LayerNorm legacy-key patch applied. DeBERTa-v3 LayerNorms will load correctly.')


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score, accuracy_score, classification_report
import json, time, gc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
assert device.type == 'cuda', 'GPU not available! Runtime > Change runtime type > GPU (T4)'

## 2. Load CONDA dataset

Upload the three CONDA CSV files: `CONDA_train.csv`, `CONDA_valid.csv`, `CONDA_test.csv` (Ctrl+click to select all three at once).

In [ ]:
from google.colab import files
uploaded = files.upload()

train_df = pd.read_csv('CONDA_train.csv')
valid_df = pd.read_csv('CONDA_valid.csv')
print(f'Train: {train_df.shape}, Valid: {valid_df.shape}')
train_df.head()

## 3. Preprocessing

- Map intent labels `E/I/A/O` → integers `0/1/2/3`
- Sort by `conversationId` and `chatTime` so prior messages are well-defined
- Load DeBERTa-v3-base tokenizer

In [ ]:
LABEL2ID = {'E': 0, 'I': 1, 'A': 2, 'O': 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

train_df['label'] = train_df['intentClass'].map(LABEL2ID)
valid_df['label'] = valid_df['intentClass'].map(LABEL2ID)

# Sort so context lookup is correct
for df in (train_df, valid_df):
    df.sort_values(['conversationId', 'chatTime'], inplace=True, kind='mergesort')
    df.reset_index(drop=True, inplace=True)

print('Label distribution (train):')
print(train_df['label'].value_counts().sort_index().rename(index=ID2LABEL))

In [ ]:
MODEL_NAME = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Sanity check: speaker tags like 'P3:' should tokenize cleanly
test = 'P3: ez mid [SEP] P7: report him'
print(f'Tokenization of {test!r}:')
print(' ', tokenizer.tokenize(test))

## 4. Contextualized dataset (k=0)

**k=0**: just the target message, no context. This is the baseline.


In [ ]:
MAX_K = 10  # we pre-compute up to 10 even if K is smaller

def build_context_index(df, max_k=10):
    ctx = [[] for _ in range(len(df))]
    for _, group in df.groupby('conversationId', sort=False):
        idxs = group.index.tolist()
        for pos, i in enumerate(idxs):
            ctx[i] = idxs[max(0, pos - max_k):pos]
    return ctx

train_context_idx = build_context_index(train_df, max_k=MAX_K)
valid_context_idx = build_context_index(valid_df, max_k=MAX_K)
n_with_ctx = sum(1 for c in train_context_idx if c)
print(f'Train rows with at least 1 prior msg: {n_with_ctx}/{len(train_df)} ({100*n_with_ctx/len(train_df):.1f}%)')

In [ ]:
SEP = ' [SEP] '

class CONDAContextDataset(Dataset):
    """
    Builds: 'P3: prev1 [SEP] P7: prev2 [SEP] ... [SEP] P3: target'
    k=0 => target alone (no speaker tag) — matches the original baseline.
    """
    def __init__(self, df, context_idx, tokenizer, k, max_length=256):
        self.df = df
        self.context_idx = context_idx
        self.tokenizer = tokenizer
        self.k = k
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def _fmt(self, row, include_speaker):
        text = str(row['utterance'])
        return f"P{int(row['playerSlot'])}: {text}" if include_speaker else text

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if self.k == 0:
            input_str = self._fmt(row, include_speaker=False)
        else:
            ctx_rows = self.context_idx[idx][-self.k:]
            if ctx_rows:
                parts = [self._fmt(self.df.iloc[j], True) for j in ctx_rows]
                parts.append(self._fmt(row, True))
                input_str = SEP.join(parts)
            else:
                input_str = self._fmt(row, True)
        enc = self.tokenizer(
            input_str, padding='max_length', truncation=True,
            max_length=self.max_length, return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(int(row['label']), dtype=torch.long),
        }

MAX_LENGTH = 128 if K == 0 else 256
train_dataset = CONDAContextDataset(train_df, train_context_idx, tokenizer, K, MAX_LENGTH)
valid_dataset = CONDAContextDataset(valid_df, valid_context_idx, tokenizer, K, MAX_LENGTH)

# Show what a sample input looks like
demo_i = next((i for i, c in enumerate(train_context_idx) if len(c) >= min(K,3)), 0)
sample = train_dataset[demo_i]
decoded = tokenizer.decode(sample['input_ids'], skip_special_tokens=False)
print(f'Sample input (k={K}, label={ID2LABEL[sample["labels"].item()]}):')
print(decoded[:400])

## 5. Load DeBERTa-v3-base + define metrics

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    torch_dtype=torch.float32,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded ({n_params:,} parameters)')

## 5.5 · Verify the LayerNorm patch worked

Sanity check that the LayerNorm parameters are NOT at their default values.
If you see weight stats `mean ~ 1.0  std ~ 0.0` and bias stats `mean ~ 0.0
std ~ 0.0`, the patch did NOT take effect and your training will fail again.

You want to see weight values that look like a learned distribution (mean
somewhere between 0.5 and 1.0, with non-trivial std).


In [ ]:
# Sanity check: are the LayerNorm parameters loaded from pretraining,
# or are they at PyTorch's default (weight=1.0, bias=0.0)?
import torch

sample_keys = [
    'deberta.embeddings.LayerNorm.weight',
    'deberta.embeddings.LayerNorm.bias',
    'deberta.encoder.layer.0.attention.output.LayerNorm.weight',
    'deberta.encoder.layer.6.output.LayerNorm.weight',
    'deberta.encoder.layer.11.output.LayerNorm.weight',
]

state = model.state_dict()
print(f'{"parameter":<60s} {"mean":>8s} {"std":>8s}  {"loaded?":>10s}')
print('-' * 92)
ok = True
for k in sample_keys:
    if k not in state:
        print(f'{k:<60s} MISSING')
        ok = False
        continue
    v = state[k]
    mean, std = v.mean().item(), v.std().item()
    is_default = abs(mean - 1.0) < 1e-4 and std < 1e-4
    loaded = 'NO (default)' if is_default else 'YES'
    if is_default and 'weight' in k:
        ok = False
    print(f'{k:<60s} {mean:>8.4f} {std:>8.4f}  {loaded:>10s}')

if ok:
    print('\nPatch is working — LayerNorms loaded from pretraining.')
else:
    print('\nWARNING: LayerNorms appear to be at default values. Patch did not take effect.')


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1_per = f1_score(labels, preds, average=None, zero_division=0, labels=[0,1,2,3])
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_E': f1_per[0],
        'f1_I': f1_per[1],
        'f1_A': f1_per[2],
        'f1_O': f1_per[3],
    }

## 6. Train

Checkpoints save to Google Drive. If Colab disconnects, rerun this cell and it picks up where it left off.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Resume from existing checkpoint if any
ckpts = [p for p in CKPT_DIR.glob('checkpoint-*') if p.is_dir()]
resume = bool(ckpts)
if resume:
    print(f'Resuming from existing checkpoint(s): {[p.name for p in ckpts]}')

t0 = time.time()
train_result = trainer.train(resume_from_checkpoint=resume)
elapsed_min = (time.time() - t0) / 60
print(f'\nTraining done in {elapsed_min:.1f} min')
print(f'Final train loss: {train_result.training_loss:.4f}')

## 7. Final evaluation on the validation set

In [ ]:
eval_metrics = trainer.evaluate()
print('Validation metrics (best model):')
for kk, v in eval_metrics.items():
    if isinstance(v, float):
        print(f'  {kk:30s} {v:.4f}')

In [ ]:
# Detailed per-class report
preds_output = trainer.predict(valid_dataset)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids
print(classification_report(
    y_true, y_pred,
    target_names=['E', 'I', 'A', 'O'],
    digits=4, zero_division=0,
))

## 7.5 · Inline classification test (no download needed)

Run this cell to classify a handful of sample messages directly inside the
notebook. This is the fastest way to verify the model actually works before
spending time downloading and redeploying it.

Expected (roughly):
- explicit insults (`"fuck"`, `"noob retard"`)  →  E
- implicit / sarcasm (`"ez 500"`, `"gg ez"`)    →  I  (often, harder)
- action commands (`"report him"`, `"pause"`)   →  A
- neutral chat (`"thanks"`, `"hello"`)          →  O


In [ ]:
# ===== Inline classification test =====
# Uses the trainer's best model (already loaded in `trainer.model` after
# load_best_model_at_end=True). Sends raw text the same way the k=0 dataset
# does at training time (no speaker prefix, no context).

import torch

device = next(trainer.model.parameters()).device
trainer.model.eval()

def classify(text):
    enc = tokenizer(text, return_tensors='pt', truncation=True,
                    max_length=MAX_LENGTH, padding='max_length').to(device)
    with torch.no_grad():
        logits = trainer.model(**enc).logits[0]
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    pred_idx = int(probs.argmax())
    return ID2LABEL[pred_idx], float(probs[pred_idx]), probs

SAMPLES = [
    # (text, expected_class)
    ('fuck',                  'E'),
    ('noob retard',           'E'),
    ('shit team',             'E'),
    ('ez 500',                'I'),
    ('gg ez',                 'I'),
    ('team is bad lol',       'I'),
    ('report him',            'A'),
    ('pause please',          'A'),
    ('wait',                  'A'),
    ('thanks for the help',   'O'),
    ('hello team',            'O'),
    ('good game',             'O'),
]

correct = 0
print(f'{"text":<28s} {"pred":>5s} {"true":>5s}  conf    | E      I      A      O')
print('-' * 78)
for text, expected in SAMPLES:
    pred, conf, probs = classify(text)
    flag = '✓' if pred == expected else '✗'
    if pred == expected:
        correct += 1
    print(f'{text!r:<28s} {pred:>5s} {expected:>5s}  {conf:>5.1%}  | '
          f'{probs[0]:>5.1%} {probs[1]:>5.1%} {probs[2]:>5.1%} {probs[3]:>5.1%}  {flag}')

print(f'\n{correct}/{len(SAMPLES)} samples agree with intuition. '
      f'(Note: implicit toxicity at k=0 is genuinely hard — '
      f'misses on "ez 500"/"gg ez" don\'t mean the patch failed.)')


## 8. Save model + metadata to Drive

Final model goes to `{MODEL_DIR}`. The demo website loads it via:
```python
AutoModelForSequenceClassification.from_pretrained('/path/to/k0_model')
```

In [ ]:
# Save best model + tokenizer in HuggingFace format
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

# Save metrics + config that the demo site can read
meta = {
    'k': K,
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'train_minutes': elapsed_min,
    'final_train_loss': float(train_result.training_loss),
    'eval_metrics': {kk: float(v) for kk, v in eval_metrics.items() if isinstance(v, (int, float))},
    'label2id': LABEL2ID,
    'id2label': ID2LABEL,
    'sep_token': SEP.strip(),
    'speaker_tag_format': 'P{playerSlot}:',
}
with open(MODEL_DIR / 'meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Saved model to {MODEL_DIR}')
print('Contents:')
for p in sorted(MODEL_DIR.iterdir()):
    size_mb = p.stat().st_size / 1e6
    print(f'  {p.name:30s} {size_mb:8.2f} MB')

In [ ]:
# Optional: delete intermediate checkpoints to free Drive space (comment out if you want to keep them)
import shutil
for ckpt in CKPT_DIR.glob('checkpoint-*'):
    if ckpt.is_dir():
        shutil.rmtree(ckpt)
        print(f'Removed {ckpt}')

## Done

The model for **k=0** is saved at `{MODEL_DIR}`.

Next: run the notebook for the next k value, then the demo website can compare predictions across all trained models.

In [ ]:
# Save the trained model to Colab local disk (bypasses Drive entirely), then download
import tarfile, json, os
from google.colab import files

LOCAL_MODEL_DIR = f'/content/k{K}_model'
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

# Save to /content (Colab's local disk, ~100GB free, NOT Drive)
trainer.save_model(LOCAL_MODEL_DIR)
tokenizer.save_pretrained(LOCAL_MODEL_DIR)

# Save metadata
meta = {
    'k': K,
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'label2id': LABEL2ID,
    'id2label': ID2LABEL,
    'sep_token': SEP.strip(),
    'speaker_tag_format': 'P{playerSlot}:',
}
# Include eval metrics if they're in scope
try:
    meta['eval_metrics'] = {kk: float(v) for kk, v in eval_metrics.items() if isinstance(v, (int, float))}
except NameError:
    pass

with open(f'{LOCAL_MODEL_DIR}/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved to', LOCAL_MODEL_DIR)
print('Contents:')
for p in sorted(os.listdir(LOCAL_MODEL_DIR)):
    size_mb = os.path.getsize(f'{LOCAL_MODEL_DIR}/{p}') / 1e6
    print(f'  {p:30s} {size_mb:8.2f} MB')

# Archive and download
archive_path = f'/content/k{K}_model.tar.gz'
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(LOCAL_MODEL_DIR, arcname=f'k{K}_model')

print(f'\nArchive: {archive_path} ({os.path.getsize(archive_path)/1e6:.1f} MB)')
print('Starting download...')
files.download(archive_path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('/content/k0_model.tar.gz', '/content/drive/MyDrive/k0_model.tar.gz')
print('Copied to Drive')